In [1]:

import os
import sys
import json
import numpy as np
import wikipediaapi


# プロジェクトのutils追加
project_root = os.path.join(os.path.dirname(__file__), "..") # os.path.dirname(__file__): スクリプト自身のパス
sys.path.append(project_root)


NameError: name '__file__' is not defined

In [ ]:

print_results = True  # get_concept_list_with_complete_cos_similar_termsの結果を表示するかどうか

def fetch_wikipedia_page(title: str, lang: str = "en") -> dict:
    """Fetch a Wikipedia page by title and language.
    wiki pageを取得する．
    Args:
        title (str): Wikipediaページのタイトル
        lang (str): Wikipediaの言語コード（例: "en", "ja"）

    Returns:
        dict: ページ情報を含む辞書．ページが存在しない場合は'exists'キーがFalseとなる．

    Note:
        * titleに完全一致するページのみが取得されるため，google検索のように最も近いが異なるものを検索結果として返すことはない．
        * そのため，提示した固有名詞とは違う情報のページが返されることは無い．
        * また，titleの綴りが間違っていても, userがよく間違う綴りの場合は, wikipedia側で正しいページにリダイレクトしてくれる．
            * 例: "Colloseum" -> "Colosseum"のページを取得
    
    """
    wiki = wikipediaapi.Wikipedia(
        user_agent="my-wikipedia-fetcher/1.0 (contact: your_email@example.com)",
        language=lang,
    )
    page = wiki.page(title)

    if not page.exists():
        return {"exists": False, "title": title, "lang": lang}

    return {
        "exists": True,
        "title": page.title,
        "url": page.fullurl,
        "summary": page.summary,
        "text": page.text,  # 全文（長いので注意）
    }


def main():

    wikiget_target_concepts = []

    print("=== Start Fetching Wikipedia Pages ===")
    # *** wikipediaページを取得して保存 ***
    wikipage_savedir = os.path.join(project_root, "data", "wikipages")
    os.makedirs(wikipage_savedir, exist_ok=True)

    # 既に保存されているwikipageを確認
    existing_wikipages = set()
    for filename in os.listdir(wikipage_savedir):
        if filename.endswith(".json"):
            concept = filename.split('.json')[0].replace('_', ' ')
            existing_wikipages.add(concept)

    # 各conceptに対してWikipediaページを取得し保存
    for concept in wikiget_target_concepts:
        # 既に保存されている場合はスキップ
        if concept in existing_wikipages:
            print(f"【Skipped】 Wikipedia page for '{concept}' already exists.")
            continue

        result = fetch_wikipedia_page(concept, lang="en")
        if not result["exists"]:
            print(f"【Skipped】 Wikipedia page for '{concept}' does not exist.")
            continue

        filename = f"{concept.replace(' ', '_')}.json"
        with open(os.path.join(wikipage_savedir, filename), "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
    
        print(f"【Saved】 Wikipedia page for '{concept}' saved as '{filename}'.")

    print("=== Finished Fetching Wikipedia Pages ===")


if __name__ == "__main__":
    main()

In [ ]:

def fetch_wikipedia_page(title: str, lang: str = "en") -> dict:
    """Fetch a Wikipedia page by title and language.
    wiki pageを取得する．
    Args:
        title (str): Wikipediaページのタイトル
        lang (str): Wikipediaの言語コード（例: "en", "ja"）

    Returns:
        dict: ページ情報を含む辞書．ページが存在しない場合は'exists'キーがFalseとなる．

    Note:
        * titleに完全一致するページのみが取得されるため，google検索のように最も近いが異なるものを検索結果として返すことはない．
        * そのため，提示した固有名詞とは違う情報のページが返されることは無い．
        * また，titleの綴りが間違っていても, userがよく間違う綴りの場合は, wikipedia側で正しいページにリダイレクトしてくれる．
            * 例: "Colloseum" -> "Colosseum"のページを取得
    
    """
    wiki = wikipediaapi.Wikipedia(
        user_agent="my-wikipedia-fetcher/1.0 (contact: your_email@example.com)",
        language=lang,
    )
    page = wiki.page(title)

    if not page.exists():
        print(f"{title} does not exist in Wikipedia ({lang}).")
        return {"exists": False, "title": title, "lang": lang}

    return {
        "exists": True,
        "title": page.title,
        "url": page.fullurl,
        "summary": page.summary,
        "text": page.text,  # 全文（長いので注意）
    }

import re

def extract_wiki_main_text(text: str) -> str:
    """Wikipediaページの情報から本文テキストのみを抽出する。
    Args:
        text (str): Wikipediaページの全文

    Returns:
        str: Wikipediaページの本文テキスト。ページが存在しない場合は空文字列を返す。
    """

    STOP_SECTIONS = {
        "see also",
        "references",
        "notes",
        "further reading",
        "external links",
        "bibliography",
        "sources",
        "works cited",
        "citations",
    }
    # 見出し候補を | で連結
    titles = "|".join(re.escape(x) for x in STOP_SECTIONS)

    # 行頭に単独で出る見出しを検出
    # 例:
    # See also
    # References
    #
    # または wiki風:
    # == See also ==
    pattern = rf"(?mi)^\s*(?:=+\s*)?(?:{titles})(?:\s*=+)?\s*$"

    # 最初に見つかった見出し以降を全て削除
    m = re.search(pattern, text)
    if m:
        text = text[:m.start()]

    # 脚注番号 [1], [23] を削除
    text = re.sub(r"\[\d+\]", "", text)

    # 空行を整理
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text



In [13]:
wiki_info = fetch_wikipedia_page("Sieve of Eratosthenes", lang="en")
wiki_info

{'exists': True,
 'title': 'Sieve of Eratosthenes',
 'url': 'https://en.wikipedia.org/wiki/Sieve_of_Eratosthenes',
 'summary': "In mathematics, the sieve of Eratosthenes is an ancient algorithm for finding all prime numbers up to any given limit.\nIt does so by iteratively marking as composite (i.e., not prime) the multiples of each prime, starting with the first prime number, 2. The multiples of a given prime are generated as a sequence of numbers starting from that prime, with constant difference between them that is equal to that prime. This is the sieve's key distinction from using trial division to sequentially test each candidate number for divisibility by each prime. Once all the multiples of each discovered prime have been marked as composites, the remaining unmarked numbers are primes.\nThe earliest known reference to the sieve (Ancient Greek: κόσκινον Ἐρατοσθένους, kóskinon Eratosthénous) is in Nicomachus of Gerasa's Introduction to Arithmetic, an early 2nd-century CE book wh

In [14]:
print(wiki_info["text"])


In mathematics, the sieve of Eratosthenes is an ancient algorithm for finding all prime numbers up to any given limit.
It does so by iteratively marking as composite (i.e., not prime) the multiples of each prime, starting with the first prime number, 2. The multiples of a given prime are generated as a sequence of numbers starting from that prime, with constant difference between them that is equal to that prime. This is the sieve's key distinction from using trial division to sequentially test each candidate number for divisibility by each prime. Once all the multiples of each discovered prime have been marked as composites, the remaining unmarked numbers are primes.
The earliest known reference to the sieve (Ancient Greek: κόσκινον Ἐρατοσθένους, kóskinon Eratosthénous) is in Nicomachus of Gerasa's Introduction to Arithmetic, an early 2nd-century CE book which attributes it to Eratosthenes of Cyrene, a 3rd-century BCE Greek mathematician, though describing the sieving by odd numbers i

In [19]:
text = extract_wiki_main_text(wiki_info["text"])
print(text)

In mathematics, the sieve of Eratosthenes is an ancient algorithm for finding all prime numbers up to any given limit.
It does so by iteratively marking as composite (i.e., not prime) the multiples of each prime, starting with the first prime number, 2. The multiples of a given prime are generated as a sequence of numbers starting from that prime, with constant difference between them that is equal to that prime. This is the sieve's key distinction from using trial division to sequentially test each candidate number for divisibility by each prime. Once all the multiples of each discovered prime have been marked as composites, the remaining unmarked numbers are primes.
The earliest known reference to the sieve (Ancient Greek: κόσκινον Ἐρατοσθένους, kóskinon Eratosthénous) is in Nicomachus of Gerasa's Introduction to Arithmetic, an early 2nd-century CE book which attributes it to Eratosthenes of Cyrene, a 3rd-century BCE Greek mathematician, though describing the sieving by odd numbers i

In [20]:
# text = wiki_info["text"]

# 本文中のconceptを<unused>に置換する. 小文字大文字は区別しない
import re
text = re.sub(r'\bSieve of Eratosthenes\b', '<unused>', text, flags=re.IGNORECASE)
print(text)

In mathematics, the <unused> is an ancient algorithm for finding all prime numbers up to any given limit.
It does so by iteratively marking as composite (i.e., not prime) the multiples of each prime, starting with the first prime number, 2. The multiples of a given prime are generated as a sequence of numbers starting from that prime, with constant difference between them that is equal to that prime. This is the sieve's key distinction from using trial division to sequentially test each candidate number for divisibility by each prime. Once all the multiples of each discovered prime have been marked as composites, the remaining unmarked numbers are primes.
The earliest known reference to the sieve (Ancient Greek: κόσκινον Ἐρατοσθένους, kóskinon Eratosthénous) is in Nicomachus of Gerasa's Introduction to Arithmetic, an early 2nd-century CE book which attributes it to Eratosthenes of Cyrene, a 3rd-century BCE Greek mathematician, though describing the sieving by odd numbers instead of by 